# 🧠 Attention Mechanisms
**Course:** Deep Learning  
**Lecture:** 06 — Attention Mechanisms and Transformers  
**Part:** a — Attention Mechanism

---

> **Where we are:** Sequence-to-sequence models with RNNs struggle with long contexts. Attention allows models to focus on relevant parts of the input dynamically.

**What you'll learn:**
- Additive, Multiplicative, and Scaled Dot-Product Attention
- The mathematical foundations of Attention
- How scaling prevents softmax saturation
- Masking techniques

**Exam relevance:** ⭐⭐⭐⭐⭐ (Understanding attention is critical for all modern NLP questions)


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12


## 📖 Theory: Three Major Attention Variants

1. **Additive Attention (Bahdanau, 2014)**
   - Formula: $e_{ij} = v^T \tanh(W_1 h_i + W_2 s_j)$
   - Uses a feed-forward network to learn alignment. Flexible but parameter-heavy and slower.

2. **Multiplicative / Dot-Product Attention (Luong, 2015)**
   - Formula: $e_{ij} = h_i^T s_j$
   - Uses simple dot product. Fast and parameter-free, but variance grows with dimension.

3. **Scaled Dot-Product Attention (Vaswani, 2017 — Transformers)**
   - Formula: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$
   - $Q$: Query (what we are looking for)
   - $K$: Key (what we match against)
   - $V$: Value (what we extract)
   - Divides by $\sqrt{d_k}$ to prevent softmax saturation in high dimensions. This is the foundation of all Transformers.


## 🔀 Your Options

| Option | Pros | Cons | When to use |
|---|---|---|---|
| Additive | Highly expressive | Slower, extra weights | Complex, low-resource tasks |
| Multiplicative | Extremely fast | Unstable with large $d_k$ | Small dimension embeddings |
| Scaled Dot-Product | Fast, stable | Assumes specific alignment | Transformers, standard NLP |

At this step, you can choose between: Additive, Multiplicative, or Scaled Dot-Product. We will implement Scaled Dot-Product as it is the most ubiquitous.


In [ ]:
# ✏️ Your Turn: Numpy Scaled Dot-Product Attention

def numpy_scaled_dot_product_attention(Q, K, V):
    # ### YOUR CODE HERE ###
    # 1. Compute dot product of Q and K transposed
    # 2. Divide by sqrt(d_k)
    # 3. Apply softmax
    # 4. Multiply by V
    
    # HINT: K.shape is (seq_len, d_k), so you need K.T to multiply with Q
    # HINT: softmax formulation: exp(x) / sum(exp(x), axis=-1, keepdims=True)
    pass

# Dummy data
Q_dummy = np.random.randn(3, 4)
K_dummy = np.random.randn(3, 4)
V_dummy = np.random.randn(3, 4)


In [ ]:
# ✅ Solution: Numpy Scaled Dot-Product Attention

def numpy_scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    
    # 1. Q K^T
    scores = np.dot(Q, K.T)
    
    # 2. Scale
    scaled_scores = scores / np.sqrt(d_k)
    
    # 3. Softmax
    exp_scores = np.exp(scaled_scores - np.max(scaled_scores, axis=-1, keepdims=True)) # stability
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # 4. Output
    output = np.dot(attention_weights, V)
    
    return output, attention_weights

out, weights = numpy_scaled_dot_product_attention(Q_dummy, K_dummy, V_dummy)
print("Output shape:", out.shape)
print("Weights sum to 1 along rows:", np.allclose(np.sum(weights, axis=-1), 1.0))


In [ ]:
# ✏️ Your Turn: Masking and TensorFlow implementation

def tf_scaled_dot_product_attention(q, k, v, mask=None):
    # ### YOUR CODE HERE ###
    # Implement the same logic in TensorFlow
    # If mask is provided, add (mask * -1e9) to the scaled scores before softmax
    pass


In [ ]:
# ✅ Solution: TF Implementation with Masking

def tf_scaled_dot_product_attention(q, k, v, mask=None):
    matmul_qk = tf.matmul(q, k, transpose_b=True)
    
    d_k = tf.cast(tf.shape(k)[-1], tf.float32)
    scaled_attention_logits = matmul_qk / tf.math.sqrt(d_k)
    
    if mask is not None:
        scaled_attention_logits += (mask * -1e9)  
        
    attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
    output = tf.matmul(attention_weights, v)
    
    return output, attention_weights

# Causal Mask (look-ahead mask)
seq_len = 4
causal_mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
print("Causal Mask:\n", causal_mask.numpy())


In [ ]:
# 🔬 Comparison: Heatmaps and Scaling effects

q = tf.random.normal((1, 5, 64))
k = tf.random.normal((1, 5, 64))
v = tf.random.normal((1, 5, 64))

# 1. Without scaling (Multiplicative)
logits_unscaled = tf.matmul(q, k, transpose_b=True)
weights_unscaled = tf.nn.softmax(logits_unscaled, axis=-1)

# 2. With scaling
_, weights_scaled = tf_scaled_dot_product_attention(q, k, v)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(weights_unscaled[0], ax=axes[0], cmap='viridis', vmin=0, vmax=1)
axes[0].set_title("Without Scaling (Saturated)")
sns.heatmap(weights_scaled[0] if len(weights_scaled.shape)==3 else weights_scaled, ax=axes[1], cmap='viridis', vmin=0, vmax=1)
axes[1].set_title("With sqrt(d_k) Scaling")
plt.show()


## 🎓 Exam Corner

### Question 1 (Conceptual)
Write the scaled dot-product attention formula and explain what Q, K, V represent. Why do we divide by $\sqrt{d_k}$? What happens if we don't?

<details><summary>💡 Answer</summary>

**Formula:** $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$

- **Q (Query):** The current element we are processing, looking for relevant info.
- **K (Key):** The elements we match against to compute relevance scores.
- **V (Value):** The actual content we extract once we have the scores.

We divide by $\sqrt{d_k}$ to prevent the dot products from growing too large. If we don't, the softmax function will be pushed into regions where it has extremely small gradients (saturation), making learning very slow.

</details>

### Question 2 (Practical)
What is the shape of the attention weight matrix given Q: (seq_len, d_k) and K: (seq_len, d_k)? What is the difference between self-attention and cross-attention?

<details><summary>💡 Answer</summary>

**Shape:** `(seq_len, seq_len)` - It's a square matrix where each row represents a query token and each column represents a key token.

**Difference:**
- **Self-attention:** Q, K, and V all come from the exact same sequence.
- **Cross-attention:** Q comes from the target/decoder sequence, while K and V come from the source/encoder sequence.

</details>
